<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h1>Day 4 · Streaming, quality and governance</h1><p>Receive Kafka events with a persistent checkpoint; reconcile delivery and event identities; validate, quarantine and recheck a batch; document quality and governance decisions.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h1>اليوم 4 · التدفق والجودة والحوكمة</h1><p>استقبل أحداث Kafka مع نقطة تحقق مستمرة، وطابق سجلات الوصول وهويات الأحداث، وافحص الدفعة واعزل المعيب وأعد الفحص، ووثّق قرارات الجودة والحوكمة.</p></td></tr></table>



<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>1. Continue your project</h2><p>Use the same repository and successful Day 1 workspace. Restore your handoff ZIP at the repository root when using a new session. Read <a href="README.md">today’s guide</a> before running all cells in order.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>١. استكمل مشروعك</h2><p>استخدم المستودع نفسه ومساحة اليوم الأول الناجحة. استعد ملف الانتقال في جذر المستودع عند استخدام جلسة جديدة. اقرأ <a href="README.md">دليل اليوم</a> ثم شغّل الخلايا بالترتيب.</p></td></tr></table>



In [ ]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'course.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the complete course repository; see docs/SETUP.md.')
sys.path.insert(0, str(ROOT / 'src'))
SOURCE = ROOT / 'data/masar-small-v1'
from masar.workspace import require_fixed_dataset, completed_bronze_workspace
from masar.runtime import require_environment, start_spark
from masar.native_contracts import validate_stage_result
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)
print('Continue workspace:', WORK.relative_to(ROOT))


<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>05 · Receive Kafka events</h2><p>Follow <a href="labs/lab05/WALKTHROUGH.md">the lab walkthrough</a>. The cell runs real operations and saves reports; inspect the checks and explain one observation in your lab notes.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>05 · استقبل أحداث Kafka</h2><p>اتبع <a href="labs/lab05/WALKTHROUGH.md">شرح اللاب</a>. تنفذ الخلية العمليات وتحفظ تقاريرها؛ افحص النتائج وفسّر ملاحظة واحدة في ملف اللاب.</p></td></tr></table>



In [ ]:
from masar.streaming import run_stream_lab
spark = start_spark(WORK, kafka=True)
try:
    result = run_stream_lab(spark, SOURCE, WORK)
    validate_stage_result('lab05_streaming', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print('Transport rows:', [phase['transport_rows'] for phase in result['phases']])
    print('Unique event IDs:', [phase['unique_event_ids'] for phase in result['phases']])
    spark.read.format('delta').load(str(WORK/result['event_table'])).select('event_id','trip_id','event_ts').orderBy('event_id').show(5, truncate=False)
finally:
    spark.stop()


<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>06 · Validate and quarantine</h2><p>Follow <a href="labs/lab06/WALKTHROUGH.md">the lab walkthrough</a>. The cell runs real operations and saves reports; inspect the checks and explain one observation in your lab notes.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>06 · افحص واعزل السجلات</h2><p>اتبع <a href="labs/lab06/WALKTHROUGH.md">شرح اللاب</a>. تنفذ الخلية العمليات وتحفظ تقاريرها؛ افحص النتائج وفسّر ملاحظة واحدة في ملف اللاب.</p></td></tr></table>



In [ ]:
from masar.quality_gate import run_quality_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_quality_lab(spark, SOURCE, WORK)
    validate_stage_result('lab06_quality', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print('Quarantined records:')
    spark.read.format('delta').load(str(WORK/result['quarantine_table'])).show(7, truncate=False)
    print('Approved rows:', spark.read.format('delta').load(str(WORK/result['approved_table'])).count())
finally:
    spark.stop()


<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Review and save</h2><p>Answer <a href="PRACTICE.md">the questions</a> as part of the existing lab notes, then use <a href="COMPLETION.md">the completion checklist</a>. Save your notebook with actual outputs. The archive below is a handoff, not another assignment.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>راجع واحفظ</h2><p>أجب عن <a href="PRACTICE.md">الأسئلة</a> ضمن ملاحظات اللاب، ثم استخدم <a href="COMPLETION.md">قائمة الاكتمال</a>. احفظ دفترك بالمخرجات الفعلية. ملف الانتقال أدناه ليس تكليفًا آخر.</p></td></tr></table>



In [ ]:
from pathlib import Path
import zipfile
pointer = ROOT / 'outputs/day01_bronze_success.json'
archive = ROOT / 'outputs/day04_handoff.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(pointer, pointer.relative_to(ROOT).as_posix())
    for path in sorted(WORK.rglob('*')):
        if path.is_file():
            bundle.write(path, path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print('Retain the notebook outputs, notes and', archive.relative_to(ROOT))
